# 00 · Setup: environment, data, checkpoints

Run this once. Every result in the paper comes from a run in the campaign registry (`repro/campaign.json`),
e.g. `r0_s0` (the undefended predictor, seed 0) or `e1_R3_s1` (attacks on the patched predictor R3, seed 1).
For each run the repo stores the exact configuration it used (`repro/configs/<key>.yaml`); the GitHub release
**`checkpoints-v1`** holds every model those configurations need (195 files, 19.3 GB, sha256 in
`repro/checkpoints.json`) and the per-run outputs (`run_results.tar.gz`).

Notebook folders (see `README.md` for the full index):

* `training/` — one notebook per trained model; launches the training as Slurm jobs.
* `tables/`, `figures/`, `results/` — one notebook per table, figure and reported result. They never train:
  `SOURCE="recompute"` re-runs the evaluation from the released checkpoints on GPUs,
  `SOURCE="released"` only aggregates the original run outputs (CPU, seconds).

## Environment
```bash
bash scripts/install_kernel.sh   # once: kernel "hackpatch" = project venv + the scripts/env.sh variables
```
Pick the kernel "hackpatch (scripts/env.sh)" in Jupyter. Non-interactive:
`NB_SOURCE=released jupyter nbconvert --to notebook --execute --inplace notebooks/tables/table1_defences.ipynb`.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
import torch
print("torch", torch.__version__, "| cuda visible:", torch.cuda.is_available())
print("scratch runs dir:", repro.RUNS)
print("artifacts dir   :", repro.ARTIFACTS)

torch 2.10.0+cu128 | cuda visible: False
scratch runs dir: /fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability
artifacts dir   : /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release


## Data
Audio corpora are not redistributed. Manifests (CSV: `utt_id, path, mos, system, corpus, split`) are built by
`scripts/build_eval_manifests.py` (BVCC, NISQA, SOMOS, TMHINT-QI, VMC'23, URGENT) and `scripts/build_vbd_manifests.py`
(VoiceBank-DEMAND); sources are listed in `docs/eval_corpora.md`.

In [3]:
from hackpatch.data.manifests import manifest_path
needed = ["bvcc/train", "bvcc/val", "bvcc/test", "bvcc_ood/test", "nisqa/test_for", "nisqa/test_p501",
          "nisqa/test_livetalk", "somos/test", "tmhintqi/test", "vmc23/track1a", "vmc23/track1b", "vmc23/track2",
          "vmc23/track3", "urgent24_sqa/test", "urgent25_sqa/test"]
pd.DataFrame([{"manifest": s, "exists": manifest_path(*s.split("/")).exists()} for s in needed])

,manifest,exists
0,bvcc/train,True
1,bvcc/val,True
2,bvcc/test,True
3,bvcc_ood/test,True
4,nisqa/test_for,True
5,nisqa/test_p501,True
6,nisqa/test_livetalk,True
7,somos/test,True
8,tmhintqi/test,True
9,vmc23/track1a,True


## Checkpoints and released outputs
`repro.fetch()` downloads every checkpoint and verifies its sha256 (on the original cluster it symlinks the
original files instead). Other notebooks fetch only what they need. The release is private: a GitHub token is read
from `GITHUB_TOKEN` or the git credential store.

In [4]:
ckpts = repro.fetch()
pd.DataFrame(repro.manifest()["assets"]).groupby("role").agg(
    files=("asset", "size"), GB=("size", lambda s: round(s.sum() / 1e9, 2)))

[fetch] 195 checkpoints ready in /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints


,files,GB
role,,
attacker,128,0.79
loop_predictor,36,13.59
referenced,3,0.03
selected,28,4.86


In [5]:
runs_dir = repro.fetch_results()
pd.Series({k: v["kind"] for k, v in repro.registry().items()}).value_counts().rename("runs")

e2                   30
e1                   24
e3_grpo              16
baseline              9
ablation              6
bench                 4
hnp                   3
r0                    3
e3_eval               3
multispace            3
e1_attacker_seeds     3
e3_eval_r2            1
Name: runs, dtype: int64